# 프로젝트 4: 에너지관리 - 에너지 원단위 & 효율 분석

## 프로젝트 배경

당신은 **한국정밀산업(주)** 에너지관리팀의 데이터 분석가입니다.  
최근 전기요금이 대폭 인상되면서 경영진이 **에너지 절감 방안**을 긴급히 요구하고 있습니다.

> *"올해 전기요금이 작년 대비 25% 올랐습니다.  
> A라인의 전력 소비가 유독 높은데, 원인이 뭔가요?  
> C라인은 3월에 인버터를 도입했는데 효과가 있나요?  
> 대기전력 낭비와 피크 시간대 전력 집중 문제도 분석해 주세요.  
> 구체적인 절감 금액까지 산출해 주시면 좋겠습니다."*

### 분석 목표
1. **전력 사용 패턴 분석** - 시간대별·요일별·라인별 전력 사용 패턴 파악
2. **에너지 원단위 분석** - 생산량 대비 에너지 효율 (kWh/개) 비교
3. **대기전력 분석** - 비가동 시 낭비되는 전력 규모 산출
4. **전기요금 분석** - TOU(시간대별 차등 요금) 기반 비용 분석
5. **절감 전략 수립** - 구체적인 절감 포인트와 기대 절감액 도출

### 데이터 설명

| 파일 | 설명 | 주요 컬럼 |
|------|------|----------|
| `p4_equipment.csv` | 설비 에너지 프로파일 (12대) | rated_power_kw, has_inverter, standby_power_kw, energy_grade |
| `p4_energy_log.csv` | 시간별 전력 사용 (~52,000건) | timestamp, power_kwh, is_operating, ambient_temp_c |
| `p4_production_log.csv` | 일별 생산 실적 (~1,275건) | production_qty, operating_hours, defect_qty |
| `p4_tariff.csv` | 전기 요금 단가표 (TOU) | season, time_zone, rate_won_kwh |

### 전기 요금 체계 (TOU: Time of Use)

| 시간대 | 춘추계 (1~5, 9~12월) | 하계 (6~8월) |
|--------|---------------------|-------------|
| 경부하 (23:00~09:00) | 60원/kWh | 65원/kWh |
| 중간부하 (09~10, 12~13, 17~23시) | 85원/kWh | 105원/kWh |
| 최대부하 (10~12, 13~17시) | 120원/kWh | 150원/kWh |

### 에너지 원단위란?

```
에너지 원단위 = 에너지 사용량 (kWh) / 생산량 (개)

  - 원단위가 낮을수록 에너지 효율이 높음
  - 같은 제품을 적은 에너지로 생산 = 효율적
  - 원단위 개선 = 비용 절감 + 탄소 감축
```

---

## Part 0: 환경 설정 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

equip = pd.read_csv('../data/project4/p4_equipment.csv', encoding='utf-8-sig')
energy = pd.read_csv('../data/project4/p4_energy_log.csv', encoding='utf-8-sig')
prod = pd.read_csv('../data/project4/p4_production_log.csv', encoding='utf-8-sig')
tariff = pd.read_csv('../data/project4/p4_tariff.csv', encoding='utf-8-sig')

energy['timestamp'] = pd.to_datetime(energy['timestamp'])
prod['date'] = pd.to_datetime(prod['date'])

print('데이터 로드 완료!')
print(f'설비: {len(equip)}건 | 에너지: {len(energy):,}건 | 생산: {len(prod):,}건 | 요금: {len(tariff)}건')

---
## Part 1: 데이터 탐색 및 전처리 (15점)

에너지 데이터는 시간별로 수집되며, 센서 통신 장애 등으로 결측치가 포함됩니다.  
생산 데이터와 결합하여 에너지 원단위를 분석하기 위해 적절한 전처리가 필요합니다.

### 문제 1-1: 데이터 탐색 (5점)

1. 4개 데이터프레임의 기본 정보(shape, dtypes)를 확인하세요
2. `energy` 데이터의 결측치 현황을 컬럼별로 확인하세요
3. `energy` 데이터의 기간(시작~끝), 설비별 건수를 확인하세요
4. `equip` 테이블에서 라인별 설비 현황(정격전력, 인버터, 대기전력, 에너지등급)을 확인하세요
5. `tariff` 요금 단가표를 확인하세요

In [ ]:
# 여기에 코드 작성


### 문제 1-2: 결측치 처리 (5점)

1. `energy['power_kwh']`: 설비별 시간순 보간(interpolate) → 잔여 ffill/bfill
2. `energy['ambient_temp_c']`: 시간순 보간 (전체 동일 외기온도)
3. `prod['production_qty']`: 같은 설비의 평균값으로 대체
4. `prod['operating_hours']`: 같은 설비의 평균값으로 대체
5. 처리 후 결측치 0 확인

In [ ]:
# 여기에 코드 작성


### 문제 1-3: 파생 컬럼 추가 (5점)

1. `energy`에 파생 컬럼 추가:
   - `date` (날짜), `hour` (시간), `month` (월), `weekday` (요일, 0=월~6=일)
   - `line` (설비ID에서 추출: A라인/B라인/C라인)
   - `time_zone` (경부하/중간부하/최대부하 - TOU 시간대 기준)
   - `tariff_rate` (시간대별 요금 단가, 원/kWh)

2. `energy`에 **전기요금 컬럼** 추가:
   - `cost_won` = power_kwh × tariff_rate

TOU 시간대 분류 기준:
```python
# 경부하: 23시~09시 (23, 0, 1, ..., 8)
# 최대부하: 10~11시, 13~16시
# 중간부하: 그 외 (9시, 12시, 17~22시)
```

In [ ]:
# 여기에 코드 작성


---
## Part 2: 시간대별 전력 사용 패턴 (25점)

에너지 절감의 첫 단계는 **"언제, 어디서, 얼마나"** 전력을 사용하는지 파악하는 것입니다.  
시간대별·라인별 패턴을 분석하면 낭비 포인트가 보입니다.

> **현업 포인트**: 비가동 시간(야간/주말)에도 높은 전력이 관찰되면 → 대기전력 낭비

### 문제 2-1: 라인별·시간대별 전력 사용 히트맵 (7점)

1. 라인별·시간(0~23시)별 평균 전력(kWh)을 집계하세요
2. **히트맵** (3행×24열)으로 시각화하세요 (Figure 16×5)
   - x축: 시간(0~23), y축: 라인(A/B/C)
   - 색상: 전력 사용량 (높을수록 빨강)
3. 어떤 라인의 어떤 시간대가 전력 소비가 가장 높은가?
4. 야간(22~06시) 전력이 높은 라인은?

In [ ]:
# 여기에 코드 작성


### 문제 2-2: 가동 vs 비가동 전력 비교 - 대기전력 분석 (6점)

1. 설비별로 **가동 시 평균 전력**과 **비가동 시 평균 전력(대기전력)**을 비교하세요
2. **대기전력 비율** = 비가동 전력 / 가동 전력 × 100 (%)을 계산하세요
3. 설비별 대기전력을 수평 바 차트로 시각화 (라인별 색상 구분)
4. 대기전력이 가장 높은 설비 3대와 `equip` 테이블의 스펙(인버터, 에너지등급)을 비교하세요

> **절감 포인트**: 대기전력 비율이 15% 이상이면 절전모드 도입 효과가 큼

In [ ]:
# 여기에 코드 작성


### 문제 2-3: 월별 전력 추이 - C라인 인버터 도입 효과 (6점)

C라인은 2024년 3월에 인버터를 도입했습니다.

1. **라인별 월별** 평균 전력(kWh)을 라인 차트로 시각화하세요 (Figure 14×6)
2. C라인의 **도입 전(1~2월)** vs **도입 후(3~6월)** 평균 전력을 비교하세요
3. 통계적으로 유의미한 차이인지 **t-검정**으로 확인하세요 (p-value < 0.05?)
4. **절감률(%)** = (도입전 - 도입후) / 도입전 × 100 을 계산하세요
5. C라인 비가동 시 전력도 개선되었는지 확인하세요 (대기전력 절감)

In [ ]:
# 여기에 코드 작성


### 문제 2-4: 요일별·시간대별 패턴 - 주말 대기전력 (6점)

1. **요일별**(월~일) 평균 전력을 바 차트로 시각화하세요
2. **요일×시간대** 히트맵 (7행×24열)을 만드세요
3. 주말(토·일) 총 전력 사용량을 계산하고, 그 중 **대기전력 비중**을 산출하세요
4. 외기온도(ambient_temp_c)와 전력 사용량의 **상관관계**를 산점도 + 회귀선으로 시각화하세요
   - 여름(5~6월)과 봄(1~4월)을 색상으로 구분

In [ ]:
# 여기에 코드 작성


---
## Part 3: 에너지 원단위 분석 (25점)

에너지 원단위는 **생산 효율의 에너지 관점 지표**입니다.  
같은 제품을 적은 에너지로 만들수록 원단위가 낮아집니다.

```
에너지 원단위 (kWh/개) = 일별 총 전력사용량 / 일별 총 생산량
```

> **현업 포인트**: 원단위는 단순 전력량보다 중요합니다.  
> 전력을 많이 쓰더라도 생산량이 더 많으면 효율적일 수 있습니다.

### 문제 3-1: 에너지 원단위 계산 (7점)

1. `energy`에서 **설비별·일별** 전력 합계를 집계하세요
2. `prod`의 **설비별·일별** 생산량 합계와 결합(merge)하세요
   - 컴프레서(EQ-x04)는 생산 직접 안 하므로 제외
3. **에너지 원단위 = daily_power_kwh / daily_production_qty**를 계산하세요
4. 원단위의 기본 통계(평균, 중앙값, 표준편차)를 확인하세요
5. 원단위 분포를 **히스토그램**으로 시각화하세요 (라인별 색상 구분)

In [ ]:
# 여기에 코드 작성


### 문제 3-2: 라인별·제품별 원단위 비교 (6점)

1. **라인별** 평균 원단위를 바 차트로 비교하세요
2. **설비별** 평균 원단위를 수평 바 차트로 시각화 (라인별 색상)
3. **제품별** 평균 원단위를 비교하세요
4. 가장 비효율적인 라인-설비-제품 조합은?

In [ ]:
# 여기에 코드 작성


### 문제 3-3: 원단위 월별 추이 - C라인 개선 효과 (6점)

1. **라인별 월별** 평균 원단위를 라인 차트로 시각화하세요
2. C라인의 인버터 도입 전(1~2월) vs 후(3~6월) 원단위를 비교하세요
3. 원단위 개선률(%) = (도입전 - 도입후) / 도입전 × 100
4. B라인의 원단위를 벤치마크(목표)로 설정하고,  
   A라인이 B라인 수준으로 개선하면 **연간 절감 가능한 전력량(kWh)**을 산출하세요

In [ ]:
# 여기에 코드 작성


### 문제 3-4: 원단위 vs 가동률 관계 (6점)

1. 설비별·일별 **가동률** = 가동시간 / 24 × 100 (%)을 계산하세요  
   (`prod`의 operating_hours 사용)
2. 원단위와 가동률의 **산점도** + 회귀선을 그리세요 (라인별 색상)
3. 상관계수를 구하고, 가동률이 높을수록 원단위가 개선되는지 분석하세요
4. 가동률이 낮은 날(<50%)에 원단위가 급격히 나빠지는 현상이 보이는지 확인하세요  
   (비가동 시간의 대기전력이 원단위를 악화시키는 효과)

In [ ]:
# 여기에 코드 작성


---
## Part 4: 전기요금 & 절감 분석 (20점)

TOU(Time of Use) 요금 체계에서는 **"언제" 전력을 사용하느냐**에 따라 비용이 크게 달라집니다.  
피크 시간대(최대부하)에 집중된 사용을 분산하면 같은 전력량으로도 요금을 줄일 수 있습니다.

> **현업 포인트**: 전기요금 = Σ(시간별 전력 × 시간대별 단가)  
> 최대부하 시간대 단가는 경부하의 2배 이상!

### 문제 4-1: 시간대별 요금 구조 분석 (5점)

1. 전체 기간 **시간대별(경부하/중간부하/최대부하)** 총 전력량과 총 비용을 집계하세요
2. 시간대별 **비중(%)** - 전력량 비중 vs 비용 비중을 비교하세요
3. 2개 바 차트(전력 비중, 비용 비중)를 나란히 시각화 (Figure 14×5)
4. 최대부하 시간대의 전력 비중 vs 비용 비중 차이를 코멘트하세요

In [ ]:
# 여기에 코드 작성


### 문제 4-2: 라인별·월별 전기요금 분석 (5점)

1. **라인별·월별** 전기요금 합계를 집계하세요
2. **누적 막대 차트**로 시각화 (x축: 월, 색상: 라인)
3. 전체 기간 **총 전기요금**과 **라인별 비중**을 산출하세요
4. 월별 요금 추이에서 여름(6월)과 봄(1~4월)의 차이를 분석하세요

In [ ]:
# 여기에 코드 작성


### 문제 4-3: 대기전력 비용 산출 - 절감 잠재량 (5점)

1. 비가동(is_operating=False) 시간대의 **총 전력량**과 **총 비용**을 산출하세요
2. **설비별** 대기전력 비용을 산출하고, 상위 5대를 확인하세요
3. 만약 A라인에 절전모드를 도입하여 대기전력을 **50% 절감**한다면,  
   **연간 절감 예상 금액(원)**은 얼마인가? (6개월 데이터 × 2)
4. 대기전력 절감 대상 설비 우선순위를 정하세요

In [ ]:
# 여기에 코드 작성


### 문제 4-4: 피크 시프팅(Peak Shifting) 절감 시뮬레이션 (5점)

최대부하 시간대의 전력 일부를 경부하 시간대로 이전(shift)하면 요금을 절감할 수 있습니다.

1. 최대부하 시간대 전력의 **10%를 경부하로 이전**한다고 가정
2. 이전 전/후의 **총 전기요금 차이**를 계산하세요
3. **20%, 30% 이전** 시나리오도 계산하여 비교 표를 만드세요
4. 절감액과 현실적 실행 가능성을 코멘트하세요

In [ ]:
# 여기에 코드 작성


---
## Part 5: 에너지 절감 대시보드 & 전략 제안 (15점)

### 문제 5-1: 에너지 분석 대시보드 (8점)

**2행 3열 (6패널)** 대시보드를 만드세요. Figure 크기: (20, 12)

| 위치 | 차트 | 내용 |
|------|------|------|
| (1,1) | 그룹 바 차트 | 라인별 월별 전력 사용량 |
| (1,2) | 히트맵 | 시간(0~23) × 요일(월~일) 평균 전력 |
| (1,3) | 수평 바 차트 | 설비별 대기전력 (인버터 유무 색상 구분) |
| (2,1) | 라인 차트 | 라인별 월별 원단위 추이 |
| (2,2) | 파이 차트 | 시간대별 전기요금 비중 |
| (2,3) | 산점도 | 외기온도 vs 전력 사용량 (라인별 색상) |

전체 제목: '에너지 사용 현황 대시보드 (2024년 상반기)'

In [ ]:
# 여기에 코드 작성


### 문제 5-2: 에너지 절감 전략 제안 (7점)

분석 결과를 바탕으로 아래 항목을 **마크다운 셀에** 작성하세요.

1. **현재 에너지 사용 현황 요약**: 라인별 전력량, 요금, 원단위 핵심 수치
2. **3대 낭비 포인트**:
   - 대기전력 낭비 (규모와 대상 설비)
   - 피크 시간대 집중 (비용 비중)
   - 라인별 효율 격차 (A라인 vs B라인)
3. **C라인 인버터 도입 효과**: 절감률, 통계적 유의성
4. **구체적 절감 방안과 기대 절감액**:
   - 방안1: A라인 절전모드 도입 → 연간 ○○만원
   - 방안2: 피크 시프팅 (○○%) → 연간 ○○만원
   - 방안3: A라인 인버터 도입 → 연간 ○○만원
5. **추진 우선순위**: 투자비 대비 효과(ROI) 기준 순위

### 분석 결론 (여기에 작성)

1. **현재 에너지 사용 현황**: 

2. **3대 낭비 포인트**: 

3. **C라인 인버터 효과**: 

4. **절감 방안**: 

5. **추진 우선순위**: 


---
## 수고하셨습니다!

### 학습 체크리스트
- [ ] 시간별 에너지 데이터 전처리 및 파생 컬럼 생성
- [ ] 시간대별·요일별 전력 사용 패턴 히트맵 시각화
- [ ] 가동/비가동 전력 비교로 대기전력 분석
- [ ] 에너지 원단위 계산 및 라인별·제품별 비교
- [ ] C라인 인버터 도입 전후 효과 검증 (t-검정)
- [ ] TOU 요금 체계 기반 전기요금 산출
- [ ] 대기전력 절감 잠재량 산출
- [ ] 피크 시프팅 시뮬레이션
- [ ] 6패널 에너지 대시보드 구성
- [ ] 데이터 기반 에너지 절감 전략 제안